# Optimisation PySpark Avancée

Ce notebook explore les techniques d'optimisation PySpark pour le traitement distribué des données Amazon Reviews.

## Objectifs
- Stratégies de partitioning (hash, range, custom)
- Caching intelligent et persistance
- Broadcast variables pour optimiser les joins
- Monitoring Spark UI et identification des goulots

## Métriques à Optimiser
- Temps de traitement des requêtes
- Utilisation mémoire et CPU
- Shuffle operations minimisées
- Data skew évité

In [ ]:
# Import des bibliothèques
import sys
sys.path.append('../src')

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import time
import pandas as pd
import matplotlib.pyplot as plt
from utils.spark_session import create_spark_session

# Créer la session Spark optimisée
spark = create_spark_session("spark_optimization")

print("Session Spark créée avec configuration optimisée")
print(f"Version: {spark.version}")
print(f"App Name: {spark.sparkContext.appName}")

In [ ]:
# Configuration Spark avancée
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes": "128MB",
    "spark.sql.adaptive.localShuffleReader.enabled": "true",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.sql.shuffle.partitions": "200",
    "spark.default.parallelism": "200",
    "spark.sql.autoBroadcastJoinThreshold": "10MB",
    "spark.sql.broadcastTimeout": "300"
}

# Appliquer la configuration
for key, value in config.items():
    spark.conf.set(key, value)
    
print("Configuration Spark appliquée:")
for key, value in config.items():
    print(f"  {key}: {value}")

In [ ]:
# Charger les données nettoyées
def load_cleaned_data(sample_size=50000):
    """Charger un échantillon des données nettoyées"""
    electronics_path = "../data/processed/amazon_reviews_electronics_clean.json"
    
    # Charger avec schéma optimisé
    schema = StructType([
        StructField("reviewerID", StringType(), False),
        StructField("asin", StringType(), False),
        StructField("reviewerName", StringType(), True),
        StructField("helpful", ArrayType(IntegerType()), True),
        StructField("reviewText", StringType(), True),
        StructField("overall", DoubleType(), False),
        StructField("summary", StringType(), True),
        StructField("unixReviewTime", LongType(), False),
        StructField("reviewTime", TimestampType(), True),
        StructField("review_length", IntegerType(), True),
        StructField("review_word_count", IntegerType(), True),
        StructField("helpful_votes", IntegerType(), True),
        StructField("total_votes", IntegerType(), True),
        StructField("helpful_ratio", DoubleType(), True),
        StructField("review_date", TimestampType(), True),
        StructField("review_year", IntegerType(), True),
        StructField("review_month", IntegerType(), True),
        StructField("review_dayofweek", IntegerType(), True),
        StructField("rating_category", StringType(), True)
    ])
    
    df = spark.read.json(electronics_path, schema=schema)
    
    # Échantillonner si nécessaire
    if sample_size and df.count() > sample_size:
        df = df.sample(False, sample_size / df.count(), seed=42)
    
    print(f"Chargé {df.count():,} enregistrements")
    return df

# Charger les données
df = load_cleaned_data(50000)
df.printSchema()
df.show(3)

## 1. Stratégies de Partitioning

In [ ]:
def benchmark_partitioning(df, partition_col, num_partitions):
    """Benchmark différentes stratégies de partitioning"""
    print(f"\n=== Benchmark Partitioning: {partition_col} ({num_partitions} partitions) ===")
    
    # Mesurer le temps de partitioning
    start_time = time.time()
    
    if partition_col == "hash":
        # Hash partitioning sur reviewerID
        df_partitioned = df.repartition(num_partitions, "reviewerID")
    elif partition_col == "range":
        # Range partitioning sur overall
        df_partitioned = df.repartitionByRange(num_partitions, "overall")
    elif partition_col == "custom":
        # Custom partitioning basé sur rating_category
        df_partitioned = df.repartition(num_partitions, "rating_category")
    else:
        df_partitioned = df.repartition(num_partitions)
    
    partition_time = time.time() - start_time
    
    # Forcer l'évaluation
    df_partitioned.count()
    
    # Analyser la distribution des partitions
    partition_stats = df_partitioned.rdd.mapPartitionsWithIndex(
        lambda idx, it: [(idx, len(list(it)))]
    ).collect()
    
    print(f"Temps de partitioning: {partition_time:.2f}s")
    print(f"Nombre de partitions: {df_partitioned.rdd.getNumPartitions()}")
    print(f"Distribution des partitions (tailles): {[stat[1] for stat in partition_stats[:10]]}")
    print(f"Taille moyenne par partition: {df_partitioned.count() / df_partitioned.rdd.getNumPartitions():.0f}")
    
    return df_partitioned, partition_time

# Tester différentes stratégies
strategies = [
    ("hash", "reviewerID", 50),
    ("range", "overall", 20),
    ("custom", "rating_category", 10)
]

results = []
for strategy, col, parts in strategies:
    df_part, time_taken = benchmark_partitioning(df, strategy, parts)
    results.append((strategy, time_taken))

# Visualiser les résultats
plt.figure(figsize=(10, 6))
strategies_names = [r[0] for r in results]
times = [r[1] for r in results]

plt.bar(strategies_names, times)
plt.title('Performance des Stratégies de Partitioning')
plt.xlabel('Stratégie')
plt.ylabel('Temps (secondes)')
plt.show()

## 2. Caching Intelligent et Persistance

In [ ]:
def benchmark_caching_strategies(df):
    """Benchmark différentes stratégies de caching"""
    print("=== Benchmark Caching Strategies ===")
    
    # Préparer une requête complexe
    complex_query = df.groupBy("reviewerID").agg(
        F.count("*").alias("num_reviews"),
        F.avg("overall").alias("avg_rating"),
        F.stddev("overall").alias("rating_std")
    ).filter(F.col("num_reviews") > 1)
    
    strategies = [
        ("no_cache", None),
        ("memory_only", "MEMORY_ONLY"),
        ("memory_and_disk", "MEMORY_AND_DISK"),
        ("memory_and_disk_ser", "MEMORY_AND_DISK_SER")
    ]
    
    results = []
    
    for strategy_name, storage_level in strategies:
        print(f"\n--- Test: {strategy_name} ---")
        
        # Créer une copie pour le test
        df_test = df.alias(f"df_{strategy_name}")
        query_test = df_test.groupBy("reviewerID").agg(
            F.count("*").alias("num_reviews"),
            F.avg("overall").alias("avg_rating"),
            F.stddev("overall").alias("rating_std")
        ).filter(F.col("num_reviews") > 1)
        
        # Appliquer le cache si nécessaire
        if storage_level:
            if storage_level == "MEMORY_ONLY":
                query_test.cache()
            elif storage_level == "MEMORY_AND_DISK":
                query_test.persist(StorageLevel.MEMORY_AND_DISK)
            elif storage_level == "MEMORY_AND_DISK_SER":
                query_test.persist(StorageLevel.MEMORY_AND_DISK_SER)
        
        # Premier exécution (cache miss)
        start_time = time.time()
        result1 = query_test.count()
        first_time = time.time() - start_time
        
        # Deuxième exécution (cache hit si applicable)
        start_time = time.time()
        result2 = query_test.count()
        second_time = time.time() - start_time
        
        print(f"Première exécution: {first_time:.2f}s")
        print(f"Deuxième exécution: {second_time:.2f}s")
        print(f"Speedup: {first_time/second_time:.2f}x" if second_time > 0 else "Speedup: N/A")
        
        results.append({
            'strategy': strategy_name,
            'first_time': first_time,
            'second_time': second_time,
            'speedup': first_time/second_time if second_time > 0 else 0
        })
        
        # Nettoyer le cache
        query_test.unpersist()
    
    return pd.DataFrame(results)

# Exécuter les benchmarks
from pyspark.storagelevel import StorageLevel
caching_results = benchmark_caching_strategies(df)

# Visualiser les résultats
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Temps d'exécution
x = range(len(caching_results))
width = 0.35

ax1.bar([i - width/2 for i in x], caching_results['first_time'], width, label='Première exécution')
ax1.bar([i + width/2 for i in x], caching_results['second_time'], width, label='Deuxième exécution')
ax1.set_xlabel('Stratégie de Cache')
ax1.set_ylabel('Temps (secondes)')
ax1.set_title('Temps d\'Exécution par Stratégie')
ax1.set_xticks(x)
ax1.set_xticklabels(caching_results['strategy'])
ax1.legend()

# Speedup
ax2.bar(caching_results['strategy'], caching_results['speedup'])
ax2.set_xlabel('Stratégie de Cache')
ax2.set_ylabel('Speedup (x)')
ax2.set_title('Speedup par Stratégie de Cache')

plt.tight_layout()
plt.show()

## 3. Broadcast Variables pour Joins Optimisés

In [ ]:
def benchmark_join_strategies(df):
    """Benchmark différentes stratégies de join"""
    print("=== Benchmark Join Strategies ===")
    
    # Créer un petit dataframe de référence (produits populaires)
    popular_products = df.groupBy("asin").agg(
        F.count("*").alias("review_count")
    ).orderBy(F.col("review_count").desc()).limit(100)
    
    print(f"Produits populaires: {popular_products.count()}")
    
    # Stratégie 1: Join normal
    print("\n--- Join Normal ---")
    start_time = time.time()
    join_normal = df.join(popular_products, "asin", "inner")
    result_normal = join_normal.count()
    time_normal = time.time() - start_time
    print(f"Temps: {time_normal:.2f}s, Résultats: {result_normal:,}")
    
    # Stratégie 2: Broadcast join
    print("\n--- Broadcast Join ---")
    start_time = time.time()
    join_broadcast = df.join(F.broadcast(popular_products), "asin", "inner")
    result_broadcast = join_broadcast.count()
    time_broadcast = time.time() - start_time
    print(f"Temps: {time_broadcast:.2f}s, Résultats: {result_broadcast:,}")
    
    # Stratégie 3: Shuffle hash join explicite
    print("\n--- Shuffle Hash Join ---")
    start_time = time.time()
    join_shuffle = df.join(
        popular_products.hint("shuffle_hash"), "asin", "inner"
    )
    result_shuffle = join_shuffle.count()
    time_shuffle = time.time() - start_time
    print(f"Temps: {time_shuffle:.2f}s, Résultats: {result_shuffle:,}")
    
    # Comparer les résultats
    results = {
        'Normal': time_normal,
        'Broadcast': time_broadcast,
        'Shuffle Hash': time_shuffle
    }
    
    # Visualiser
    plt.figure(figsize=(10, 6))
    plt.bar(results.keys(), results.values())
    plt.title('Performance des Stratégies de Join')
    plt.ylabel('Temps (secondes)')
    plt.show()
    
    # Calculer le speedup
    speedup = time_normal / time_broadcast
    print(f"\nSpeedup Broadcast vs Normal: {speedup:.2f}x")
    
    return results

# Exécuter les benchmarks
join_results = benchmark_join_strategies(df)

## 4. Optimisation des Requêtes Complexes

In [ ]:
def optimize_complex_queries(df):
    """Optimiser des requêtes analytiques complexes"""
    print("=== Optimisation de Requêtes Complexes ===")
    
    # Requête 1: Window functions sans optimisation
    print("\n--- Window Functions (Non optimisé) ---")
    start_time = time.time()
    
    window_spec = Window.partitionBy("reviewerID").orderBy(F.desc("unixReviewTime"))
    
    df_window_naive = df.withColumn(
        "row_num", F.row_number().over(window_spec)
    ).filter(F.col("row_num") <= 5)
    
    result_naive = df_window_naive.count()
    time_naive = time.time() - start_time
    print(f"Temps: {time_naive:.2f}s, Résultats: {result_naive:,}")
    
    # Requête 2: Window functions avec optimisation
    print("\n--- Window Functions (Optimisé) ---")
    start_time = time.time()
    
    # Réduire les données avant la window function
    df_filtered = df.select(
        "reviewerID", "asin", "overall", "unixReviewTime"
    ).repartition("reviewerID")
    
    df_window_opt = df_filtered.withColumn(
        "row_num", F.row_number().over(window_spec)
    ).filter(F.col("row_num") <= 5)
    
    result_opt = df_window_opt.count()
    time_opt = time.time() - start_time
    print(f"Temps: {time_opt:.2f}s, Résultats: {result_opt:,}")
    print(f"Speedup: {time_naive/time_opt:.2f}x")
    
    # Requête 3: Aggregations complexes
    print("\n--- Aggregations Complexes ---")
    start_time = time.time()
    
    # Multiple aggregations en une seule passe
    df_agg = df.groupBy("reviewerID").agg(
        F.count("*").alias("total_reviews"),
        F.avg("overall").alias("avg_rating"),
        F.stddev("overall").alias("rating_std"),
        F.min("overall").alias("min_rating"),
        F.max("overall").alias("max_rating"),
        F.sum(F.when(F.col("overall") >= 4, 1).otherwise(0)).alias("positive_reviews"),
        F.expr("percentile_approx(overall, 0.5)").alias("median_rating")
    )
    
    result_agg = df_agg.count()
    time_agg = time.time() - start_time
    print(f"Temps: {time_agg:.2f}s, Résultats: {result_agg:,}")
    
    return {
        'window_naive': time_naive,
        'window_optimized': time_opt,
        'aggregations': time_agg
    }

# Exécuter les optimisations
query_results = optimize_complex_queries(df)

## 5. Monitoring Spark UI et Identification des Goulots

In [ ]:
def analyze_spark_performance(df):
    """Analyser les performances Spark et identifier les goulots"""
    print("=== Analyse Performance Spark ===")
    
    # Obtenir les métriques du cluster
    sc = spark.sparkContext
    
    print("\n--- Configuration du Cluster ---")
    print(f"Application ID: {sc.applicationId}")
    print(f"Master: {sc.master}")
    print(f"Default Parallelism: {sc.defaultParallelism}")
    print(f"Executor Memory: {sc._conf.get('spark.executor.memory', 'N/A')}")
    
    # Analyser la distribution des données
    print("\n--- Distribution des Données ---")
    num_partitions = df.rdd.getNumPartitions()
    total_records = df.count()
    avg_partition_size = total_records / num_partitions
    
    print(f"Nombre de partitions: {num_partitions}")
    print(f"Total enregistrements: {total_records:,}")
    print(f"Taille moyenne par partition: {avg_partition_size:.0f}")
    
    # Analyser le skew
    partition_sizes = df.rdd.mapPartitions(lambda it: [len(list(it))]).collect()
    
    if partition_sizes:
        min_size = min(partition_sizes)
        max_size = max(partition_sizes)
        skew_ratio = max_size / min_size if min_size > 0 else float('inf')
        
        print(f"Taille min partition: {min_size}")
        print(f"Taille max partition: {max_size}")
        print(f"Skew ratio: {skew_ratio:.2f}")
        
        if skew_ratio > 3.0:
            print("⚠️  Data skew détecté! Considérer un meilleur partitioning.")
    
    # Analyser les plans d'exécution
    print("\n--- Analyse Plan d'Exécution ---")
    
    # Requête de test
    test_query = df.groupBy("reviewerID").agg(
        F.count("*").alias("num_reviews"),
        F.avg("overall").alias("avg_rating")
    ).filter(F.col("num_reviews") > 5)
    
    # Obtenir le plan d'exécution
    plan = test_query._jdf.queryExecution().logical()
    optimized_plan = test_query._jdf.queryExecution().optimizedPlan()
    
    print(f"Plan logique: {plan}")
    print(f"Plan optimisé: {optimized_plan}")
    
    # Mesurer l'impact des optimisations
    print("\n--- Impact des Optimisations ---")
    
    # Sans adaptive query execution
    spark.conf.set("spark.sql.adaptive.enabled", "false")
    start_time = time.time()
    result_no_adaptive = test_query.count()
    time_no_adaptive = time.time() - start_time
    
    # Avec adaptive query execution
    spark.conf.set("spark.sql.adaptive.enabled", "true")
    start_time = time.time()
    result_adaptive = test_query.count()
    time_adaptive = time.time() - start_time
    
    print(f"Sans AQE: {time_no_adaptive:.2f}s")
    print(f"Avec AQE: {time_adaptive:.2f}s")
    print(f"Speedup AQE: {time_no_adaptive/time_adaptive:.2f}x")
    
    return {
        'partitions': num_partitions,
        'avg_partition_size': avg_partition_size,
        'skew_ratio': skew_ratio if partition_sizes else 0,
        'adaptive_speedup': time_no_adaptive/time_adaptive
    }

# Analyser les performances
performance_metrics = analyze_spark_performance(df)

## 6. Recommandations d'Optimisation

In [ ]:
def generate_optimization_recommendations(performance_metrics):
    """Générer des recommandations d'optimisation"""
    print("=== Recommandations d'Optimisation ===")
    
    recommendations = []
    
    # Recommandations basées sur les métriques
    if performance_metrics['skew_ratio'] > 3.0:
        recommendations.append({
            'issue': 'Data skew élevé',
            'impact': 'Performance dégradée',
            'solution': 'Utiliser salting ou repartitioning intelligent'
        })
    
    if performance_metrics['avg_partition_size'] < 1000:
        recommendations.append({
            'issue': 'Partitions trop petites',
            'impact': 'Overhead scheduling',
            'solution': 'Réduire le nombre de partitions'
        })
    
    if performance_metrics['avg_partition_size'] > 1000000:
        recommendations.append({
            'issue': 'Partitions trop grandes',
            'impact': 'Mémoire et GC pressure',
            'solution': 'Augmenter le nombre de partitions'
        })
    
    if performance_metrics['adaptive_speedup'] < 1.5:
        recommendations.append({
            'issue': 'AQE peu efficace',
            'impact': 'Optimisation limitée',
            'solution': 'Ajuster spark.sql.adaptive.advisoryPartitionSizeInBytes'
        })
    
    # Afficher les recommandations
    for i, rec in enumerate(recommendations, 1):
        print(f"\n{i}. {rec['issue']}")
        print(f"   Impact: {rec['impact']}")
        print(f"   Solution: {rec['solution']}")
    
    if not recommendations:
        print("✅ Aucune optimisation majeure nécessaire!")
    
    return recommendations

# Générer les recommandations
recommendations = generate_optimization_recommendations(performance_metrics)

# Créer un rapport d'optimisation
optimization_report = {
    'timestamp': time.time(),
    'performance_metrics': performance_metrics,
    'recommendations': recommendations,
    'next_steps': [
        'Implémenter le partitioning optimal',
        'Configurer le caching approprié',
        'Utiliser les broadcast joins',
        'Monitorer avec Spark UI'
    ]
}

print("\n📊 Rapport d'Optimisation Spark généré")
print(f"   Skew ratio: {performance_metrics['skew_ratio']:.2f}")
print(f"   Speedup AQE: {performance_metrics['adaptive_speedup']:.2f}x")
print(f"   Recommandations: {len(recommendations)}")

In [ ]:
# Nettoyer les ressources
spark.catalog.clearCache()
spark.stop()
print("\n✅ Session Spark arrêtée et ressources nettoyées")